In [6]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

# ──────────────────────────────────────────────────────────────────────────────
# UTILS
from rdkit.Chem.MolStandardize import rdMolStandardize
normalizer = rdMolStandardize.TautomerEnumerator()
uncharger  = rdMolStandardize.Uncharger()

def embed_and_optimize(mol, ff="MMFF"):
    # 1) Standardize & uncharge
    mol = normalizer.Canonicalize(mol)
    mol = uncharger.uncharge(mol)
    # 2) Add hydrogens
    mol = Chem.AddHs(mol)
    # 3) Embed
    params = AllChem.ETKDGv3()
    params.maxAttempts     = 2000
    params.useRandomCoords = True
    if AllChem.EmbedMolecule(mol, params) < 0 and AllChem.EmbedMolecule(mol) < 0:
        return None
    # 4) Optimize
    if ff.upper() == "MMFF":
        if AllChem.MMFFHasAllMoleculeParams(mol):
            if AllChem.MMFFOptimizeMolecule(mol) != 0:
                # fallback to UFF if MMFF fails
                if AllChem.UFFOptimizeMolecule(mol) != 0:
                    return None
        else:
            if AllChem.UFFOptimizeMolecule(mol) != 0:
                return None
    else:
        if AllChem.UFFOptimizeMolecule(mol) != 0:
            return None
    return mol




def smiles_to_3d_sdfs(
    input_csv: str,
    output_dir: str = "sdf_out",
    ff: str = "MMFF",
    id_col: str = "identifier",
    smi_col: str = "canonical_smiles",
    extra_cols: list = None
):
    """
    Read input_csv, embed+optimize each SMILES, and write one .sdf per molecule.
    extra_cols: list of other CSV columns to carry into SDF tags
    """
    df = pd.read_csv(input_csv)
    os.makedirs(output_dir, exist_ok=True)
    extra_cols = extra_cols or []

    for idx, row in df.iterrows():
        ident = str(row[id_col])
        smi   = row[smi_col]
        mol   = Chem.MolFromSmiles(smi)
        if mol is None:
            print(f"[!] invalid SMILES for {ident}, skipping")
            continue

        m3d = embed_and_optimize(mol, ff=ff)
        if m3d is None:
            print(f"[!] embedding failed for {ident}, skipping")
            continue

        # set tags on the Mol itself
        for key, val in {
            id_col: ident,
            smi_col: smi,
            **{ col: row[col] for col in extra_cols if col in row }
        }.items():
            m3d.SetProp(key, str(val))
        
        # write out
        writer = Chem.SDWriter(os.path.join(output_dir, f"{ident}.sdf"))
        writer.write(m3d)
        writer.close()


    print(f"✅ Written {len(os.listdir(output_dir))} SDFs to “{output_dir}/”")


# ──────────────────────────────────────────────────────────────────────────────
# USAGE
# ──────────────────────────────────────────────────────────────────────────────

# 1) point these to your files / columns
INPUT_CSV  = "multi_target_YoudenJ_with_metadata.csv"
OUTPUT_DIR = "3d_sdffiles"
FORCEFIELD = "MMFF"   # or "UFF"
EXTRA      = ["name", "pred_score_ABL1", "pred_score_DYRK1A", "pred_score_TTBK1"]  # carry these into SDF tags

# 2) run conversion
smiles_to_3d_sdfs(
    input_csv  = INPUT_CSV,
    output_dir = OUTPUT_DIR,
    ff         = FORCEFIELD,
    id_col     = "identifier",
    smi_col    = "canonical_smiles",
    extra_cols = EXTRA
)


[16:49:47] Running Uncharger


[!] embedding failed for CNP0074467.1, skipping
[!] embedding failed for CNP0074465.1, skipping


[16:49:48] Running Uncharger
[16:49:48] Running Uncharger


[!] embedding failed for CNP0074332.1, skipping


[16:49:48] Running Uncharger
[16:49:48] Running Uncharger


...

[!] embedding failed for CNP0596445.1, skipping
[!] embedding failed for CNP0360272.0, skipping


[19:51:30] Running Uncharger
[19:51:30] Running Uncharger


[!] embedding failed for CNP0431140.1, skipping


[19:51:31] Running Uncharger


KeyboardInterrupt: 

In [7]:
# using pubchem api
import pandas as pd
import requests
import time
import os

# Input CSV should have columns 'identifier' and 'smiles'
input_csv = 'zero_z_sdfs_with_smiles.csv'
output_dir = 'sdf_files'
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(input_csv)

for idx, row in df.iterrows():
    smiles = row['smiles']
    identifier = str(row['identifier'])  # Ensure it's a string

    # Step 1: Get CID from SMILES
    cid_url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles}/cids/TXT'
    try:
        cid_resp = requests.get(cid_url, timeout=30)
    except Exception as e:
        print(f"Network error for {identifier}: {e}")
        continue

    if cid_resp.status_code != 200 or not cid_resp.text.strip():
        print(f"Could not retrieve CID for {identifier} (SMILES: {smiles})")
        continue
    cid = cid_resp.text.strip().split('\n')[0]
    print(f"SMILES: {smiles} -> CID: {cid}")

    # Step 2: Download 3D SDF
    sdf_url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/SDF?record_type=3d'
    try:
        sdf_resp = requests.get(sdf_url, timeout=60)
    except Exception as e:
        print(f"Network error for {identifier} (CID: {cid}): {e}")
        continue

    if sdf_resp.status_code == 200 and len(sdf_resp.content) > 100:
        sdf_path = os.path.join(output_dir, f"{identifier}.sdf")
        with open(sdf_path, 'wb') as f:
            f.write(sdf_resp.content)
        print(f"Saved 3D SDF for {identifier} (CID {cid}) to {sdf_path}")
    else:
        print(f"Failed to download SDF for {identifier} (CID {cid})")

    # Be polite to the server
    time.sleep(0.2)


SMILES: O=C(OC[C@H]1O[C@@H](OC2=CC(O)=C3C(=C2)O[C@H](C2=CC(O)=C(O)C(O)=C2)[C@H](OC(=O)C2=CC(O)=C(O)C(O)=C2)[C@H]3C2=C(O)C=C(O)C3=C2O[C@H](C2=CC=C(O)C(O)=C2)[C@H](OC(=O)C2=CC(O)=C(O)C(O)=C2)[C@H]3C2=C(O)C=C(O)C3=C2O[C@H](C2=CC(O)=C(O)C(O)=C2)[C@H](OC(=O)C2=CC(O)=C(O)C(O)=C2)[C@H]3C2=C(O)C=C(O)C3=C2O[C@H](C2=CC=C(O)C(O)=C2)[C@H](OC(=O)C2=CC(O)=C(O)C(O)=C2)[C@H]3C2=C(O)C=C(O[C@@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@H]3O)C3=C2O[C@H](C2=CC(O)=C(O)C(O)=C2)[C@H](OC(=O)C2=CC(O)=C(O)C(O)=C2)C3)[C@H](O)[C@@H](O)[C@@H]1O)C1=CC(O)=C(O)C(O)=C1 -> CID: 162803070
Failed to download SDF for CNP0028858.1 (CID 162803070)
SMILES: O=C(O[C@@H]1[C@@H](C2=C(O)C=C(O)C3=C2O[C@H](C2=CC(O)=C(O)C(O)=C2)[C@H](OC(=O)C2=CC(O)=C(O)C(O)=C2)[C@H]3C2=C(O)C=C(O)C3=C2O[C@H](C2=CC(O)=C(O)C(O)=C2)[C@H](OC(=O)C2=CC(O)=C(O)C(O)=C2)[C@H]3C2=C(O)C=C(O)C3=C2O[C@H](C2=CC=C(O)C(O)=C2)[C@@H](O)C3)C2=C(O)C=C(O)C([C@H]3C4=C(O)C=C(O)C([C@H]5C6=C(O)C=C(O)C([C@H]7C8=C(O)C=C(O)C=C8O[C@H](C8=CC=C(O)C=C8)[C@@H]7O)=C6O[C@H](C6=CC(O)=C(O)C(O)=C6)

In [3]:
## IMPORTANT 
# getting all 3d sdfs (multitargeted) from coconut db from 3d_zip.sdf file 
from rdkit import Chem
import pandas as pd
import os

def has_all_zero_z(mol):
    """Check if all Z coordinates in the first conformer are zero."""
    conf = mol.GetConformer()
    zs = [conf.GetAtomPosition(i).z for i in range(mol.GetNumAtoms())]
    return all(abs(z) < 1e-4 for z in zs)  # tolerance for floating point

# 1. Load your list of identifiers from CSV/table
ids_df = pd.read_csv("final_annotated.csv")  # Update with your file name
wanted_ids = set(ids_df['Molecule'].astype(str))  # Ensure all IDs are strings

# 2. Prepare output directories
output_dir = "final_3D"
zero_z_dir = "zero_z_sdfsssss"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(zero_z_dir, exist_ok=True)

# 3. Iterate through the COCONUT 3D SDF and extract matching molecules
supplier = Chem.SDMolSupplier("coconut_sdf_3d-06-2025.sdf")
found = 0
zero_z_found = 0

for mol in supplier:
    if mol is None:
        continue
    ident = mol.GetProp("IDENTIFIER") if mol.HasProp("IDENTIFIER") else None
    if ident in wanted_ids:
        if has_all_zero_z(mol):
            writer = Chem.SDWriter(os.path.join(zero_z_dir, f"{ident}.sdf"))
            zero_z_found += 1
        else:
            writer = Chem.SDWriter(os.path.join(output_dir, f"{ident}.sdf"))
            found += 1
            if found % 100 == 0:
                print(f"{found} molecules with valid 3D written...")
        writer.write(mol)
        writer.close()

print(f" Extraction complete. {found} valid 3D SDF files written to '{output_dir}/'.")
print(f"  {zero_z_found} molecules with all-zero Z coordinates written to '{zero_z_dir}/'.")


[14:45:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:48] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:48] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:48] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:49] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:50] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:51] Warning: molecule is tagged as 3D, but all Z coords are zero
[14:45:51] WARNING: not removing hydrogen atom with neighbor tha

 Extraction complete. 34 valid 3D SDF files written to 'final_3D/'.
  0 molecules with all-zero Z coordinates written to 'zero_z_sdfsssss/'.


In [ ]:
# getting 3d sdf non generated from folder to smiles
import os
import pandas as pd

# 1. Load your metadata file
meta = pd.read_csv("multi_target_YoudenJ_with_metadata.csv")  # adjust path as needed

# 2. Build a mapping from identifier to SMILES (and any other columns you want)
id_to_smiles = dict(zip(meta['identifier'].astype(str), meta['canonical_smiles']))

# 3. Prepare output
records = []

zero_z_dir = "zero_z_sdfs"
sdf_files = [f for f in os.listdir(zero_z_dir) if f.endswith(".sdf")]

for sdf_file in sdf_files:
    ident = os.path.splitext(sdf_file)[0]
    smiles = id_to_smiles.get(ident)
    if smiles is not None:
        records.append({'identifier': ident, 'smiles': smiles})
    else:
        print(f"[!] Identifier {ident} not found in metadata CSV.")

# 4. Save as CSV
out_df = pd.DataFrame(records)
out_df.to_csv("zero_z_sdfs_with_smiles.csv", index=False)
print(f"✅ Saved mapping for {len(out_df)} molecules to zero_z_sdfs_with_smiles.csv")


✅ Saved mapping for 290 molecules to zero_z_sdfs_with_smiles.csv


In [9]:
import os
from rdkit import Chem
from rdkit.Chem import Descriptors

# INPUT: folder containing your per‐molecule 3D sdf files
INPUT_DIR = "extracted_3d_sdfs"

# OUTPUT: folder to save Lipinski-passed molecules
OUTPUT_DIR = "lipinski_passed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def lipinski_pass(mol):
    """Return True if this RDKit Mol passes Lipinski's Rule of 5."""
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = Descriptors.NumHDonors(mol)
    hba  = Descriptors.NumHAcceptors(mol)
    return (mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10)

total = 0
passed = 0

for fn in os.listdir(INPUT_DIR):
    if not fn.lower().endswith(".sdf"):
        continue
    path = os.path.join(INPUT_DIR, fn)

    suppl = Chem.SDMolSupplier(path, removeHs=False)
    mols = [m for m in suppl if m is not None]
    if not mols:
        continue

    mol = mols[0]
    total += 1

    if lipinski_pass(mol):
        passed += 1
        # save this molecule into OUTPUT_DIR
        out_path = os.path.join(OUTPUT_DIR, fn)
        writer = Chem.SDWriter(out_path)
        writer.write(mol)
        writer.close()

print(f"Total molecules scanned: {total}")
print(f"Passed Lipinski RO5:   {passed}")
print(f"Fraction passed:       {passed/total:.1%}")
print(f"✅ Passed molecules saved in: {OUTPUT_DIR}")


Total molecules scanned: 12384
Passed Lipinski RO5:   7943
Fraction passed:       64.1%
✅ Passed molecules saved in: lipinski_passed


In [14]:
os.getcwd()

'/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT'

In [ ]:
#for renaming . to _
import os
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/lipinski_passed')

files = [f for f in os.listdir('.') if f.endswith('.sdf')]
for f in files:
    base, ext = os.path.splitext(f)
    new_name = base.replace('.', '_') + ext
    if f != new_name:
        try:
            os.rename(f, new_name)
            print(f"Renamed: {f} -> {new_name}")
        except Exception as e:
            print(f"Error renaming {f}: {e}")



Renamed: CNP0000023.1.sdf -> CNP0000023_1.sdf
Renamed: CNP0000152.1.sdf -> CNP0000152_1.sdf
Renamed: CNP0000184.0.sdf -> CNP0000184_0.sdf
Renamed: CNP0000185.0.sdf -> CNP0000185_0.sdf
Renamed: CNP0000300.0.sdf -> CNP0000300_0.sdf
Renamed: CNP0000337.0.sdf -> CNP0000337_0.sdf
Renamed: CNP0000353.0.sdf -> CNP0000353_0.sdf
Renamed: CNP0000377.0.sdf -> CNP0000377_0.sdf
Renamed: CNP0000422.0.sdf -> CNP0000422_0.sdf
Renamed: CNP0000427.0.sdf -> CNP0000427_0.sdf
Renamed: CNP0000555.0.sdf -> CNP0000555_0.sdf
Renamed: CNP0000594.0.sdf -> CNP0000594_0.sdf
Renamed: CNP0000597.0.sdf -> CNP0000597_0.sdf
Renamed: CNP0000604.0.sdf -> CNP0000604_0.sdf
Renamed: CNP0000614.0.sdf -> CNP0000614_0.sdf
Renamed: CNP0000619.0.sdf -> CNP0000619_0.sdf
Renamed: CNP0000627.0.sdf -> CNP0000627_0.sdf
Renamed: CNP0000648.0.sdf -> CNP0000648_0.sdf
Renamed: CNP0000667.0.sdf -> CNP0000667_0.sdf
Renamed: CNP0000890.0.sdf -> CNP0000890_0.sdf
Renamed: CNP0000929.0.sdf -> CNP0000929_0.sdf
Renamed: CNP0000972.0.sdf -> CNP00

In [ ]:
#converting sdf to pdbqt
import os
from openbabel import pybel

# Set your folder path here
folder_path = '/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/lipinski_passed'

for sdf_file in os.listdir(folder_path):
    if sdf_file.lower().endswith('.sdf'):
        sdf_path = os.path.join(folder_path, sdf_file)
        base_name = os.path.splitext(sdf_file)[0]
        print(f"Processing {sdf_file}...")

        # Read all molecules in the SDF file
        for idx, mol in enumerate(pybel.readfile("sdf", sdf_path), start=1):
            pdbqt_file = f"{base_name}_{idx}.pdbqt"
            pdbqt_path = os.path.join(folder_path, pdbqt_file)
            mol.write("pdbqt", pdbqt_path, overwrite=True)
            print(f"  Wrote {pdbqt_file}")

print("All conversions complete!")


Processing CNP0000023_1.sdf...
  Wrote CNP0000023_1_1.pdbqt
Processing CNP0000152_1.sdf...
  Wrote CNP0000152_1_1.pdbqt
Processing CNP0000184_0.sdf...
  Wrote CNP0000184_0_1.pdbqt
Processing CNP0000185_0.sdf...
  Wrote CNP0000185_0_1.pdbqt
Processing CNP0000300_0.sdf...
  Wrote CNP0000300_0_1.pdbqt
Processing CNP0000337_0.sdf...
  Wrote CNP0000337_0_1.pdbqt
Processing CNP0000353_0.sdf...
  Wrote CNP0000353_0_1.pdbqt
Processing CNP0000377_0.sdf...
  Wrote CNP0000377_0_1.pdbqt
Processing CNP0000422_0.sdf...
  Wrote CNP0000422_0_1.pdbqt
Processing CNP0000427_0.sdf...
  Wrote CNP0000427_0_1.pdbqt
Processing CNP0000555_0.sdf...
  Wrote CNP0000555_0_1.pdbqt
Processing CNP0000594_0.sdf...
  Wrote CNP0000594_0_1.pdbqt
Processing CNP0000597_0.sdf...
  Wrote CNP0000597_0_1.pdbqt
Processing CNP0000604_0.sdf...
  Wrote CNP0000604_0_1.pdbqt
Processing CNP0000614_0.sdf...
  Wrote CNP0000614_0_1.pdbqt
Processing CNP0000619_0.sdf...
  Wrote CNP0000619_0_1.pdbqt
Processing CNP0000627_0.sdf...
  Wrote C